In [1]:
import os
from getpass import getpass
from pathlib import Path
from langchain_community.document_loaders import (
    PyMuPDFLoader,
    TextLoader,
    Docx2txtLoader,
    DirectoryLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

/home/saskya/dev/tb/polylex-chatbot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path("data")

In [3]:
loaders = [
    DirectoryLoader(DATA_DIR, glob="**/*.pdf", loader_cls=PyMuPDFLoader, show_progress=True),
    DirectoryLoader(DATA_DIR, glob="**/*.txt", loader_cls=TextLoader, show_progress=True),
    DirectoryLoader(DATA_DIR, glob="**/*.docx", loader_cls=Docx2txtLoader, show_progress=True),
]

docs = []
for loader in loaders:
    docs.extend(loader.load()) # TODO : pour le moment, un doc = une page et pas de chunking supplementaire

100%|██████████| 2/2 [00:00<00:00, 114.65it/s]


In [4]:
if not os.getenv("OPENAI_API_KEY_EMBEDDINGS"):
    os.environ["OPENAI_API_KEY_EMBEDDINGS"] = getpass("Enter your RCP API key for embedding model: ")

embeddings = OpenAIEmbeddings(
    model="Qwen/Qwen3-Embedding-8B",
    base_url="https://inference.rcp.epfl.ch/v1",
    api_key=os.getenv("OPENAI_API_KEY_EMBEDDINGS")
)

In [5]:
qdrant = QdrantVectorStore.from_documents(
    docs,
    embeddings,
    url="http://localhost:6333",
    collection_name="basic_split_pages_all_formats",
)

In [6]:
query = "Comment les responsabilités en matière de contrôle interne sont-elles réparties entre la direction, les unités opérationnelles et les fonctions de supervision (audit, contrôle des risques) au sein de l'EPFL?"
found_docs = qdrant.similarity_search_with_score(query)
found_docs

[(Document(metadata={'source': 'data/41ebaf25efa8981bc6682c01e81babfc.txt', '_id': 'ddc82e6d-20b9-47d0-b070-ba222a154450', '_collection_name': 'basic_split_pages_all_formats'}, page_content="Adresse légale de l'EPFL\nCe document définit l'adresse du siège social de l'EPFL et est authentifié par acte notarié."),
  0.7737043),
 (Document(metadata={'source': 'data/209c0d50a3923b7c60605ee50d0bef78.txt', '_id': '45a2d0bf-368b-4c0b-85dd-ca9f132494fe', '_collection_name': 'basic_split_pages_all_formats'}, page_content="Ordinance on limiting admission to the École polytechnique fédérale de Lausanne\nThis ordinance defines the criteria for limiting admission, set by the ETH Board at 3000 places, to the first semester of the Bachelor's programme at the École polytechnique fédérale de Lausanne (EPFL). It applies to foreign applicants who do not have a residence permit for Switzerland (permit C) or a residence permit for European workers and their immediate families. It provides for a selection sy

In [26]:
# TODO : tester avec d'autres parametres de chunking ou voir https://docs.langchain.com/oss/python/integrations/splitters

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=20,
    separators=["\n \n", "\n\n", "\n", ". ", " ", ""],
    add_start_index=True
)

chunks = text_splitter.split_documents(docs)